In [ ]:
import os
import requests
import string
import re
import json
from dotenv import load_dotenv
from unidecode import unidecode

In [ ]:
# Read env file using python-dotenv
load_dotenv("config/.env")

In [ ]:
urls_train = ["https://www.gutenberg.org/files/1661/1661-0.txt",
              "https://www.gutenberg.org/files/834/834-0.txt",
              "https://www.gutenberg.org/files/108/108-0.txt",
              "https://www.gutenberg.org/files/2852/2852-0.txt"]
urls_eval = ["https://www.gutenberg.org/files/3289/3289-0.txt"]

os.makedirs(os.environ["TRAIN_RAW_DIR"], exist_ok=True)
os.makedirs(os.environ["EVAL_RAW_DIR"], exist_ok=True)

for url in urls_train:
  response = requests.get(url, verify=False)
  with open(f"{os.environ['TRAIN_RAW_DIR']}/{url.split('/')[-1]}", 'w', encoding='utf-8') as f:
      f.write(response.text)
for url in urls_eval:
  response = requests.get(url, verify=False)
  with open(f"{os.environ['EVAL_RAW_DIR']}/{url.split('/')[-1]}", 'w', encoding='utf-8') as f:
      f.write(response.text)

In [ ]:
class Normalizer:
    def __init__(self):
        self.punctuation_table = str.maketrans(
            "",
            "",
            string.punctuation.replace(".", "").replace("?", "").replace("!", "")
        )
        self.digits_table = str.maketrans("", "", string.digits)
        self.titles_regex = re.compile(r"\b(mr|mrs|ms)\.\s*")

    # Load raw text
    # * call load() on the training folder (TRAIN_RAW_DIR). Load all .txt files found in the folder.
    def load(self, folder_path):
        books = []
        for filename in os.listdir(folder_path):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                books.append(unidecode(f.read()))
        return books

    # Strip Gutenberg header and footer
    # * Remove all text before and including: *** START OF THE PROJECT GUTENBERG EBOOK ... ***
    # * Remove all text from and including: *** END OF THE PROJECT GUTENBERG EBOOK ... ***
    def strip_gutenberg(self, text):
        start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
        end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
        start_index = text.find(start_marker)
        # move to the end of the line after the start marker
        start_index = 0 if start_index == -1 else text.find("\n", start_index)
        end_index = text.find(end_marker)
        end_index = len(text) if end_index == -1 else end_index
        return text[start_index:end_index].strip()

    # Lowercase all text
    def lowercase(self, text):
        return text.lower()

    # Remove all punctuation
    def remove_punctuation(self, text):
        return text.translate(self.punctuation_table)

    # Remove all numbers
    def remove_numbers(self, text):
        return text.translate(self.digits_table)

    # Remove extra whitespace and blank lines
    def remove_whitespace(self, text):
        return "\n".join([line.strip() for line in text.split("\n") if line.strip()])

    # Normalize
    # * call normalize(text) which applies lowercase, remove punctuation, remove numbers, and remove extra whitespace in order.
    # * Apply all normalization steps in order: lowercase → remove punctuation → remove numbers → remove whitespace.
    # * This is the single method that other modules call to normalize text consistently.
    def normalize(self, text):
        text = self.lowercase(text)
        text = self.remove_punctuation(text)
        text = self.remove_numbers(text)
        text = self.remove_whitespace(text)
        return text

    # Word tokenize
    # * call word_tokenize(sentence) on each sentence to split it into tokens separated by a single space.
    # * Split a single sentence into a list of tokens
    def word_tokenize(self, sentence):
        return " ".join(word.strip() for word in sentence.split(" ") if word.strip())

    # Sentence tokenize
    # * split text into sentences; each becomes one line in the output file.
    def sentence_tokenize(self, text):
        text = text.replace("\n", " ")
        text = text.replace("?", ".")
        text = text.replace("!", ".")
        return "\n".join(
            [
                self.word_tokenize(sentence.strip())
                for sentence in re.sub(self.titles_regex, r"\1 ", text).split(".")
                if sentence.strip()
            ]
        )

    # Write output file
    # * concatenate all training books and write to train_tokens.txt.
    # * Format: one sentence per line, tokens separated by spaces.
    # * If implementing the Model Evaluator extra credit, also process EVAL_RAW_DIR and write eval_tokens.txt using the same pipeline.
    def save(self, sentences, filepath):
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(sentences)

In [ ]:
normalizer = Normalizer()
for mode in ["TRAIN", "EVAL"]:
    sentences = []
    books = normalizer.load(os.environ[f"{mode}_RAW_DIR"])
    for book in books:
        normalized_book = normalizer.normalize(normalizer.strip_gutenberg(book))
        sentences.append(normalizer.sentence_tokenize(normalized_book))
    normalizer.save("\n".join(sentences), f"{os.environ[f'{mode}_TOKENS']}")

In [ ]:
class NGramModel:
    def __init__(self):
        self.vocab = []
        self.model = {}

    # Build the vocabulary
    # * collect all unique words; replace any word appearing fewer than UNK_THRESHOLD times (from config/.env) with <UNK>;
    # * add <UNK> to the vocabulary.
    def build_vocab(self, token_file):
        token_freq = {}
        with open(token_file, "r", encoding="utf-8") as f:
            for sentence in f.read().splitlines():
                for token in sentence.split(" "):
                    token_freq[token] = token_freq.get(token, 0) + 1
        self.vocab = [
            token
            for token, freq in token_freq.items()
            if freq >= int(os.getenv("UNK_THRESHOLD"))
        ]
        self.vocab.append("<UNK>")

    # Build counts at all orders
    # * slide a window across every sentence and count all unique n-grams from 1-gram up to NGRAM_ORDER-gram.
    # * The number of orders is read from config/.env.
    def build_counts_and_probabilities(self, token_file):
        NGRAM_ORDER = int(os.getenv("NGRAM_ORDER"))
        self.model = {f"{i+1}gram": {} for i in range(NGRAM_ORDER)}
        with open(token_file, "r", encoding="utf-8") as f:
            for sentence in f.read().splitlines():
                tokens = [
                    tk if tk in self.vocab else "<UNK>" for tk in sentence.split(" ")
                ]
                for i in range(len(tokens)):
                    unigram = tokens[i]
                    self.model["1gram"][unigram] = (
                        self.model["1gram"].get(unigram, 0) + 1
                    )
                    for n in range(2, NGRAM_ORDER + 1):
                        if n - 1 > i:
                            continue
                        ngram = " ".join(tokens[i - n + 1 : i])
                        if ngram not in self.model[f"{n}gram"]:
                            self.model[f"{n}gram"][ngram] = {}
                        self.model[f"{n}gram"][ngram][tokens[i]] = (
                            self.model[f"{n}gram"][ngram].get(tokens[i], 0) + 1
                        )
            total_count = sum(self.model["1gram"].values())
            for unigram in self.model["1gram"]:
                self.model["1gram"][unigram] /= total_count
            for n in range(2, NGRAM_ORDER + 1):
                for ngram in self.model[f"{n}gram"]:
                    total_count = sum(self.model[f"{n}gram"][ngram].values())
                    for token in self.model[f"{n}gram"][ngram]:
                        self.model[f"{n}gram"][ngram][token] /= total_count

    # Backoff lookup
    # * try the highest-order context first, fall back to lower orders down to 1-gram.
    # * Return a dict of {word: probability} from the highest order that matches.
    # * Return empty dict if no match at any order.
    # * This is the single source of backoff logic in the project.
    def lookup(self, context):
        for n in range(int(os.getenv("NGRAM_ORDER")), 0, -1):
            if n == 1:
                return self.model["1gram"]
            tokens = context[-n + 1 :]
            ngram = " ".join(tokens)
            if ngram in self.model[f"{n}gram"]:
                return self.model[f"{n}gram"][ngram]
        return {}

    # Save all probability tables to model.json
    def save_model(self, model_path):
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        with open(model_path, "w", encoding="utf-8") as f:
            json.dump(self.model, f, ensure_ascii=False, indent=4)

    # Save vocabulary list to vocab.json
    def save_vocab(self, vocab_path):
        os.makedirs(os.path.dirname(vocab_path), exist_ok=True)
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=4)

    def load(self, model_path, vocab_path):
        with open(model_path, "r", encoding="utf-8") as f:
            self.model = json.load(f)
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)

In [ ]:
ngram_model = NGramModel()
ngram_model.build_vocab(os.environ["TRAIN_TOKENS"])
ngram_model.build_counts_and_probabilities(os.environ["TRAIN_TOKENS"])
ngram_model.save_vocab(f"{os.environ[f'VOCAB']}")
ngram_model.save_model(f"{os.environ[f'MODEL']}")

In [ ]:
ngram_model = NGramModel()
ngram_model.load(f"{os.environ[f'MODEL']}", f"{os.environ[f'VOCAB']}")

In [ ]:
class Predictor:

    # Accept a pre-loaded NGramModel and Normalizer instance. Do not load files here.
    def __init__(self, model, normalizer):
        self.model = model
        self.normalizer = normalizer

    # Call Normalizer.normalize(text);
    # * extract last NGRAM_ORDER − 1 words as context
    def normalize(self, text):
        lines = self.normalizer.normalize(text).splitlines()
        if len(lines) == 0:
            return []
        tokens = lines[-1].split(" ")
        if len(tokens) < (int(os.getenv("NGRAM_ORDER")) - 1):
            return tokens
        return tokens[-(int(os.getenv("NGRAM_ORDER")) - 1) :]

    # Replace out-of-vocabulary words with <UNK>
    def map_oov(self, context):
        return [word if word in self.model.vocab else "<UNK>" for word in context]

    # Orchestrate normalize → map_oov → NGramModel.lookup() → return top-k words sorted by probability
    def predict_next(self, text, k):
        normalized_text = self.normalize(text)
        context = self.map_oov(normalized_text)
        predictions = self.model.lookup(context)
        sorted_predictions = sorted(
            predictions.items(),
            key=lambda item: item[1],
            reverse=True,
        )
        return [pred[0] for pred in sorted_predictions[:k]]

In [ ]:
predictor = Predictor(ngram_model, normalizer)
predictor.predict_next("It was the best of times, it was the worst of", int(os.getenv("TOP_K")))